In [1]:
import sys; sys.path.append("..")
from src.bq import run

run("SELECT COUNT(*) AS n FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_20210131`")

/Users/brittneychan/ga4-funnel-analysis/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2131: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,n
0,26489


In [3]:
from src.bq import run_file, save

volume = run_file("00_exp/01_volume.sql")
save(volume, "volume")
volume

00_exp/01_volume.sql: ~0.12 GB


/Users/brittneychan/ga4-funnel-analysis/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2131: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,first_day,last_day,n_days,n_users,n_events
0,2020-11-01,2021-01-31,92,270154,4295584


In [6]:
events = run_file("00_exp/02_event_counts.sql")
save(events, "event_counts")
events.head(20)

00_exp/02_event_counts.sql: ~0.13 GB


/Users/brittneychan/ga4-funnel-analysis/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2131: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,event_name,n_events,n_users
0,page_view,1350428,269792
1,user_engagement,1058721,213004
2,scroll,493072,138098
3,view_item,386068,61252
4,session_start,354970,267116
5,first_visit,257462,257314
6,view_promotion,190104,102443
7,add_to_cart,58543,12545
8,begin_checkout,38757,9715
9,select_item,31007,13180


In [5]:
quality = run_file("00_exp/03_quality.sql")
save(quality, "data_quality")
quality.T   # one metric per row

00_exp/03_quality.sql: ~1.45 GB


/Users/brittneychan/ga4-funnel-analysis/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2131: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,0
n_events,4295584
pct_events_no_session_id,0.0
pct_events_with_session_medium,0.335045
pct_medium_obfuscated,0.212171
pct_no_device,0.0
purchase_events,5692
distinct_txn_ids,4452
purchases_missing_txn_id,906
purchases_null_revenue,0
total_revenue_usd,362165.0


 # Observations
* Dates range from 11-01-2020 to 01-31-2021 (total of 92 days).
* # Events: 4295584
* # Users: 270154
* Funnel events present (Desc # events): 
    * view item (386068)
    * add to cart (58543)
    * begin checkout (38757)
    * select item (31007)
    * add shipping info (19722)
    * add payment info (13899)
    * purchase (5692)
* Share of events without session ID: 0
* Channels use session-level soource, falling back to first-touch.
    * ~21% of events have hidden mediums (other/deleted)
* Orders = distinct transaction IDs per session, so duplicates count once. 
    * purchase events: 5692
    * distinct transaction IDs: 4452
    * purchases without transaction IDs: 906
    * ~ 330 duplicate orders
* Revenue is obfuscated sample data, so dollar figures are illustrative. Duplicate orders inflate it slightly.
